# gemma-4-developer-agent — submission builder

This notebook builds **`/kaggle/working/submission.zip`**, a declarative Google ADK
agent package. The organizers compile it with `adk-submission` and run the resulting
agent on hidden SWE-bench-style tasks with a single permitted model,
`gemma-4-e4b-it-qat-w4a16-ct`, on 4x L4 GPUs.

**What the notebook does, and nothing else:**

1. Holds the agent package as in-memory Python string literals.
2. Validates it against every submission constraint the harness enforces.
3. Writes a deterministic zip with `agent.yaml` at the archive root.

**What it deliberately does not do:** no `pip install`, no downloads, no dataset
inputs, no GPU, no network. The only dependencies are the Python standard library
and `PyYAML`, both of which are already present on Kaggle. Every byte it needs is
written inline in the cells below, so the notebook cannot drift from the package
it is supposed to emit.

## Safety design

A notebook that raises an exception scores **zero**, so the build is written as if
it will fail:

- The primary package is validated. If any single check fails, the notebook falls
  back to a **minimal known-good package** (one agent, no sub-agent, short prompt)
  that is designed to have the fewest possible ways to fail on the scorer's machine.
- If even the packaging step fails, a final cell writes an emergency minimal archive
  directly, bypassing the shared packager.
- The last cell asserts the artifact exists and is a readable zip.

## Contents

| Section | Purpose |
|---|---|
| Setup | import the standard library, resolve the working directory |
| Design rationale | why each load-bearing choice is made |
| Library | the static validator and the fallback package |
| Primary package | the agent YAML, prompts and sampling config |
| Validate | run the checks, choose primary or fallback |
| Package | write the zip and a manifest, print a report |


> **Note.** The agent package itself is what scores; this notebook is only the
> mechanism that delivers it. The interesting content is in `prompts/system.md`,
> reproduced verbatim in the cells below.

> **Baseline.** The bundled package is a byte-exact replica of the public
> `matterhorn3838/gemma-4-superagent` notebook (best public score **0.17**),
> whose `agent.yaml`, sampling and budgets are identical to the public
> **0.18** `verracodeguacas/gemma-4-agent-budget-fit-single-agent` bundle.
> Three independent public notebooks converge on this configuration
> (single agent, thinking on at budget 4096, `max_output_tokens` 8192,
> 240 s commands, 28 tool calls, 8 minutes, 80 turns); it replaced our own
> v2 bundle (public 0.08). See
> `kaggle_work/gemma-4-developer-agent/research/BASELINE_017.md` for the full
> diff and the ablation plan.


In [ ]:
"""Standard-library setup for the submission builder.

Only stdlib plus PyYAML. No pip, no network, no dataset inputs.
"""
import hashlib
import io
import json
import os
import platform
import posixpath
import sys
import textwrap
import traceback
import zipfile
from pathlib import Path

try:
    import yaml
except Exception:  # pragma: no cover - PyYAML ships with Kaggle notebooks
    yaml = None
    print("WARNING: PyYAML is unavailable; structural checks will be skipped.")


def _resolve_working() -> Path:
    """Find the directory the submission zip must be written to.

    Kaggle uses /kaggle/working. Off Kaggle (local verification) fall back to an
    override or a temporary directory, so the notebook is runnable anywhere.
    """
    for candidate in (
        os.environ.get("KAGGLE_WORKING_DIR"),
        "/kaggle/working",
        os.getcwd(),
    ):
        if not candidate:
            continue
        path = Path(candidate)
        if path.is_dir() and os.access(path, os.W_OK):
            return path
    import tempfile

    path = Path(tempfile.mkdtemp(prefix="gemma4_submission_"))
    return path


WORKING = _resolve_working()

print(f"python           {sys.version.split()[0]} ({platform.platform()})")
print(f"working directory {WORKING}")
print(f"pyyaml            {'present' if yaml is not None else 'MISSING'}")
print(f"writable          {os.access(WORKING, os.W_OK)}")


## Why the package looks the way it does

This package is a byte-exact replica of a public notebook whose best public
score is **0.17** (`matterhorn3838/gemma-4-superagent`), and whose
`agent.yaml`, sampling and budgets are identical to the public **0.18**
`verracodeguacas/gemma-4-agent-budget-fit-single-agent` bundle. Three
independent public notebooks converged on this configuration. Our previous
bundle (v2, public 0.08) differed from it in architecture, output budget,
per-task budget and prompt structure; the full diff is in
`kaggle_work/gemma-4-developer-agent/research/BASELINE_017.md`. What follows is
why each load-bearing value is what it is: the public authors' measured numbers
first, then the harness source where we have verified the mechanism ourselves
(swegemma 0.2.11 / adk_submission 0.2.13 / adk_eval_core 0.1.0).

### 1. Single agent, no sub-agents, no adapters

The 0.17/0.18 notebooks all use one `LlmAgent` with the nine harness tools. A
sub-agent's LLM calls count against the same turn budget, its `AgentTool`
schema adds a complex tenth tool choice, and the published harness study
(arXiv 2609.32459) found *general* sub-agents can hurt a small model while a
complex harness gives a weak model no gain. No LoRA adapter either: any adapter
collapses the harness KV cache from roughly 46k to roughly 7.6k tokens, and the
competition's W4A16 alias is the only permitted model.

### 2. `eval_config.yaml` — the budget is calibrated, not guessed

The scoring run gets 12 hours for ~120 sequential tasks including vLLM startup
and Phase 2 verification, and `max_time_minutes` is an `asyncio.timeout` ceiling
around the agent loop that is cancelled the instant `submit_patch` is called
(`agent_runner.py:725`, and the loop breaks on `patch_submitted` at
`agent_runner.py:920`). So the ceiling costs nothing on a task solved quickly
and only spends wall clock where a task would otherwise be killed mid-debug —
with a half-finished edit, which scores exactly zero.

| Setting | Value | Reason |
|---|---|---|
| `max_time_minutes` | 8 | measured: only about half of the fixes Gemma eventually finds are on disk after 4 scorer-speed minutes; a scored run with this cap finished inside the 12 h total |
| `max_tool_calls` | 28 | measured: runs still exploring after many calls almost never succeed; the hard cap pays for the longer time ceiling |
| `max_turns` | 80 | one LLM call per turn; more turns cannot be spent once the 28 tool calls are gone |
| `timeout_seconds` | 240 | it bounds the Phase 2 pytest run too (`verification.py`); heavy repos' test files need about 100 s, so a shorter timeout can score a correct patch as failed |

The prompt enforces the same numbers as a **call plan** (edit by call 12),
which is where the measured 87%-fail-without-edit-by-call-12 statistic is
banked.

### 3. Thinking on, 8192 output tokens

`include_thoughts: true` with `thinking_budget: 4096`; vLLM is served with
`max_model_len=32768`, and a request whose prompt plus `max_output_tokens`
exceeds that is refused and kills the session. 8192 leaves about 24k of prompt
headroom; the 16384 our previous bundle used leaves only about 16k and was
measured by the public authors to kill long tasks.

### 4. The prompt: specification first, then a call plan

The public prompt extracts the exact names, messages, types and edge cases from
the issue as a literal specification before any tool call, then locates with
one batched `git grep`, edits the root cause minimally, verifies with at most
two calls and submits. Google's Gemma function-calling guidance says the same
thing from the model side: multi-step chaining is outside Gemma's trained
distribution, so the prompt must supply the sequence explicitly; tool-use
instructions should say *when and why*, not just *what*. The graph tools are
declared but banned in the prompt: they only match exact fully-qualified ids
and usually return nothing.

### 5. Patch mechanics (verified in the harness source)

- `submit_patch` runs `git add -N .` and diffs against the `_swegemma_baseline`
  tag (`tools/execution.py:77-127`). `git add -N .` respects the repository's
  own `.gitignore`, so a new file on an ignored path silently never reaches
  the patch — the prompt's pre-submit checklist covers it.
- Protected test and runner-config files are stripped from the patch before
  grading (`harness/verification.py:62-85`), so editing them can never help.
- The working tree is graded even if `submit_patch` is never called
  (`agent_runner.py:1054-1075`); the prompt says so, and requires a non-empty
  tree.
- The instruction contains no braces except the trailing
  `{problem_description}`: ADK interpolates every brace-delimited token in an
  instruction from session state, so an accidental literal brace is a runtime
  error. The validator enforces this.

### 6. What is deliberately absent

No `skills/` directory (no repo-specific content worth disclosing on unseen
repositories), no `.py` files (never executed by the compiler, pure risk), no
files with an empty suffix (the competition allowlist omits it, so a `LICENSE`
or `Makefile` aborts the submission), no callbacks (`callback_registry` is
never passed).


## Library and fallback package

The next cell loads the static validator, and the one after that defines the
rollback package.


In [ ]:
# --- validator (spliced verbatim from src/notebook_lib/validator.py) ---------
# The source is embedded as base64 and executed rather than pasted inline: that
# keeps it byte-identical to the repository file, keeps a leading
# ``from __future__ import annotations`` legal, and avoids any escaping problem.
import base64

_SPLICED_SOURCE = '...'

VALIDATOR_SOURCE = base64.b64decode(_SPLICED_SOURCE).decode("utf-8")
VALIDATOR_LOADED = False
try:
    exec(compile(VALIDATOR_SOURCE, "validator.py", "exec"), globals())
    VALIDATOR_LOADED = True
    print(f"validator loaded ({len(VALIDATOR_SOURCE):,d} chars)")
except Exception:  # noqa: BLE001 - never fatal
    print("WARNING: the validator could not be loaded; structural checks are off.")
    print(traceback.format_exc())

if not VALIDATOR_LOADED:
    # Degraded but still functional: packaging proceeds without validation, so a
    # valid archive is produced either way. The constants the reporting cell
    # reads are restated here so no later cell can raise a NameError.
    ALLOWED_MODELS = frozenset({"gemma-4-31b-it-qat-w4a16-ct", "gemma-4-e4b-it-qat-w4a16-ct"})
    MODEL = "gemma-4-31b-it-qat-w4a16-ct"
    ALLOWED_SUFFIXES = frozenset(
        {".yaml", ".yml", ".md", ".txt", ".py", ".json", ".safetensors"}
    )
    MAX_TOTAL_SIZE_BYTES = 3 * 2**30

    class SubmissionInvalid(Exception):
        """Stand-in used when the real validator could not be imported."""

    def validate(files):
        raise SubmissionInvalid("structural validation is unavailable in this run")

    def pack(files, validate_first=True):
        stream = io.BytesIO()
        with zipfile.ZipFile(stream, "w", zipfile.ZIP_DEFLATED) as archive:
            for name in sorted(files):
                info = zipfile.ZipInfo(name, date_time=(1980, 1, 1, 0, 0, 0))
                info.compress_type = zipfile.ZIP_DEFLATED
                info.create_system = 3
                info.external_attr = 0o644 << 16
                archive.writestr(info, files[name].encode("utf-8"))
        return stream.getvalue()

    def digest(payload):
        return hashlib.sha256(payload).hexdigest()


## Primary package

The full agent package, inlined as a string mapping. This directory is the single
source of truth in the repository:

```
agent.yaml                  root agent: the 9 harness tools, no sub-agents
eval_config.yaml            the 12 hour compute budget, allocated per task
configs/sampling.yaml       generation config (thinking on, 8192 output tokens)
prompts/system.md           the agent's instructions
```

This is the byte-exact public 0.17 baseline bundle (SuperAgent), which is also
the 0.18 Budget-Fit configuration. Ablation bundles live under
`kaggle_work/gemma-4-developer-agent/bundles/` and are built into their own
notebooks with `tools/build_notebook.py --bundle ...`.


In [ ]:
PACKAGE_FILES = {
    'agent.yaml': 'name: swe_agent\nmodel: gemma-4-31b-it-qat-w4a16-ct\ninstruction: !include prompts/system.md\ntools:\n  - run_command\n  - read_file\n  - edit_file\n  - write_file\n  - get_status\n  - submit_patch\n  - get_code_neighbors\n  - search_similar_code\n  - get_code_subgraph\nskills:\n  - skills/swe-toolkit\ngenerate_content_config: !include configs/sampling.yaml\n',
    'configs/sampling.yaml': 'temperature: 0.2\ntop_p: 0.95\nmax_output_tokens: 8192\nthinking_config:\n  thinking_budget: 4096\n  include_thoughts: true\n',
    'eval_config.yaml': 'evaluation:\n  timeout_seconds: 240\n  max_tool_calls: 28\n  max_time_minutes: 8\n  max_turns: 80\n',
    'prompts/system.md': 'You are an autonomous software engineer. Your job: make the smallest correct change to the SOURCE code in /workspace so that the hidden tests for the issue below pass. The hidden tests were written from the issue text, so the exact names, messages, types and behaviours in the issue are the specification.\n\n# HARD LIMITS\n- 28 tool calls and 8 minutes in total. When either runs out, the working tree is graded as it is. An edit that exists counts, even if you never call `submit_patch`. An unedited tree always scores zero.\n- Never create, modify or delete test files (`test_*.py`, `*_test.py`, anything under `tests/`). This is an automatic failure.\n- Never run a full test suite (bare `pytest`, `pytest .`, `unittest discover`). Always name one test file, and add `-k` when you can.\n- Never look outside /workspace. All dependencies are already installed.\n- Never use `search_similar_code`, `get_code_neighbors` or `get_code_subgraph`. They return nothing useful.\n\n# CALL PLAN\n- Calls 1-8: locate and read. Calls 9-12: edit. You MUST have edited a source file by call 12. If you are unsure, make your best-guess edit in the most likely place now. A wrong edit can still be corrected. Pure reading cannot score.\n- Calls 13-20: verify and correct.\n- Call 20 onward: no new exploration. Check, fix, submit.\n- Always finish with `git diff` followed by `submit_patch`.\n\n# PROCEDURE\n\n## 1. Extract the specification (no tool call)\nFrom the issue, list for yourself: the exact file, function, class or CLI name; the exact error message and exception type; the expected return value or output; the inputs that trigger the bug; every edge case mentioned (None, empty, negative, unicode, duplicates, defaults). Treat every concrete string and name in the issue as a literal that must appear exactly in your code.\n\n## 2. Locate\n- Use the paths and symbols from the issue directly. If there is no path, run one search: `git grep -n "most specific string or identifier" -- \'*.py\' | head -20`.\n- Use `read_file` on the relevant region (about 40 lines around the target) and not whole files.\n- For a bug: find where the wrong behaviour is produced, then check how callers use it with one `git grep -n "function_name" -- \'*.py\' | head`.\n- For a feature: find the closest existing sibling (a similar function, option, class, or doc example) and copy its pattern. That pattern includes registration points such as `__all__`, option lists, dispatch tables, and imports.\n- If the issue includes a code snippet that reproduces the problem, you may run it once with `python -c` (one call, never create a file) to see the real error.\n\n## 3. Edit\n- Use `edit_file` for changes to existing code. Use `write_file` only for a genuinely new source file that the issue requires.\n- Fix the cause, not the symptom. Do not special-case the example from the issue.\n- Change only what the issue requires. Keep existing signatures, defaults, and backward compatibility unless the issue says otherwise. Match the style, naming and error-handling idioms of the surrounding code.\n- Use the exact exception class and exact message text from the issue. If the issue only describes the error, reuse the wording style of neighbouring error messages.\n- Handle every edge case you listed in step 1.\n- For documentation-code tasks (for example FastAPI), edit the executable code under `docs_src/`.\n\n## 4. Verify (at most 2 calls)\n- Find the test file with `find tests -name "*keyword*.py"`, then run `timeout 150 python -m pytest tests/path/test_x.py -x -q -k keyword`. Use `python3 -m unittest tests.test_x` for unittest projects.\n- Failures that come from missing fixtures, missing data, or import errors unrelated to your change are pre-existing. Ignore them. Never repair tests or the environment.\n- If a test related to your change fails, read the traceback, fix the source once, and rerun once. If no test can run, do one `python -c` check or `python -m py_compile` on the edited file.\n\n## 5. Final check and submit\n- Run `git status --short` and `git diff`. Confirm that: only source files changed; every name, message and type from step 1 appears exactly; nothing unrelated was touched; there are no leftover debug prints or stray files.\n- **Every file you created must be listed by `git status --short`.** The patch is built with `git add -N .`, which silently skips anything the repository\'s own `.gitignore` ignores, so a new file on an ignored path never reaches the grading diff. If a file you created is missing from the status output, run `git add -N -f <path>` and check again.\n- Delete debugging scratch files you created inside /workspace; they land in the patch. Files in /tmp never do.\n- Call `submit_patch`. Confirm `patch_size > 0` and `files_changed > 0`. If either is 0, you have not edited anything, or your edit was stripped. Edit now and submit again.\n- End with a single-sentence summary of the fix.\n\n# DECISION RULES\n- Ambiguous issue: choose the most literal reading of the issue text and act on it. Do not ask questions and do not stall.\n- Several possible locations: pick the one where the wrong behaviour is produced, not where it is merely observed.\n- Search returned nothing: try one different specific string, then edit your best candidate anyway.\n- Stuck after a failed fix: make a different, smaller change. Do not repeat the same edit.\n- Never conclude that nothing needs to change. Every task requires a non-empty source patch.\n\n\n## TOOLKIT\nUse swe.py via run_command for fast deterministic localization and verification:\n- Locate: python3 $(find / -name swe.py 2>/dev/null | head -1) locate --query "..." --root /workspace\n- Inspect: python3 $(find / -name swe.py 2>/dev/null | head -1) inspect --symbol NAME --root /workspace\n- Verify: python3 $(find / -name swe.py 2>/dev/null | head -1) verify --test "pytest args" --root /workspace\nIf run_skill_script is available, call skill_name="swe_toolkit", script_path="scripts/swe.py".\nThe toolkit auto-detects /workspace; if absent it uses the current directory.\n\n## The issue\n{problem_description}\n',
    'skills/swe-toolkit/SKILL.md': '---\nname: swe-toolkit\ndescription: "Deterministic SWE helper: locate, inspect, verify."\n---\n# swe-toolkit\nOperations: locate, inspect, verify.\nInvoke via ADK: run_skill_script(skill_name="swe-toolkit", script_path="scripts/swe.py", args=["locate","--query","..."])\nDirect: python3 $(find / -name swe.py 2>/dev/null | head -1) locate --query "..."\nBootstrap: python3 skills/swe-toolkit/scripts/bootstrap.py\nState: /tmp/swe_state.json\n',
    'skills/swe-toolkit/scripts/_index.py': 'import ast\nimport math\nimport os\nimport re\nimport sys\nfrom pathlib import Path\n\nSKIP_DIRS = {\n    ".git", ".venv", "venv", "env", ".env", "node_modules",\n    "build", "dist", "__pycache__", ".tox", ".eggs", ".mypy_cache",\n}\nMAX_BYTES = 500 * 1024\nSTOPWORDS = {\n    "the", "when", "with", "should", "error", "value", "return", "using",\n    "does", "not", "from", "this", "that", "and", "for", "are", "was",\n    "were", "has", "have", "had", "but", "can", "will", "into", "onto",\n    "than", "then", "them", "they", "their", "there", "here", "what",\n    "which", "who", "whom", "whose", "how", "why", "where", "all", "any",\n    "both", "each", "few", "more", "most", "other", "some", "such", "only",\n    "own", "same", "too", "very", "just", "now", "also", "after", "before",\n    "between", "through", "during", "without", "within", "about", "against",\n    "above", "below", "to", "of", "in", "on", "at", "by", "as", "is", "it",\n    "be", "or", "an", "a", "if", "so", "no", "up", "out", "off", "over",\n    "under", "again", "further", "once", "because", "until", "while",\n}\n\n\ndef is_binary(path):\n    try:\n        with open(path, "rb") as f:\n            chunk = f.read(8192)\n        if b"\\x00" in chunk:\n            return True\n        chunk.decode("utf-8")\n        return False\n    except Exception:\n        return True\n\n\ndef walk(root):\n    out = []\n    root = Path(root)\n    for dirpath, dirnames, filenames in os.walk(root, followlinks=False):\n        dirnames[:] = [d for d in dirnames if d not in SKIP_DIRS]\n        for fn in filenames:\n            p = Path(dirpath) / fn\n            try:\n                if p.stat().st_size > MAX_BYTES:\n                    continue\n            except Exception:\n                continue\n            if is_binary(p):\n                continue\n            out.append(p)\n    return out\n\n\ndef split_ident(tok):\n    toks = set()\n    for part in re.split(r"[_\\.]", tok):\n        if part:\n            toks.add(part.lower())\n    for m in re.findall(r"[A-Z]?[a-z]+|[A-Z]+(?=[A-Z][a-z])|[A-Z]+", tok):\n        toks.add(m.lower())\n    toks.add(tok.lower())\n    return toks\n\n\ndef tokenize(s):\n    base = re.findall(r"[A-Za-z_][A-Za-z0-9_]{2,}", s)\n    out = set()\n    for t in base:\n        if t.lower() in STOPWORDS:\n            continue\n        out |= split_ident(t)\n    return out\n\n\ndef extract_symbols(text, filepath):\n    syms = []\n    lines = text.splitlines()\n    try:\n        tree = ast.parse(text)\n    except Exception:\n        for m in re.finditer(r"^(?:async\\s+)?def\\s+([A-Za-z_]\\w*)", text, re.M):\n            syms.append({\n                "qualname": m.group(1), "filepath": filepath,\n                "start_line": None, "end_line": None,\n                "signature": m.group(0), "decorators": [], "bases": [],\n                "calls": [], "strings": [], "body": "", "is_async": m.group(0).startswith("async"),\n            })\n        for m in re.finditer(r"^class\\s+([A-Za-z_]\\w*)", text, re.M):\n            syms.append({\n                "qualname": m.group(1), "filepath": filepath,\n                "start_line": None, "end_line": None,\n                "signature": m.group(0), "decorators": [], "bases": [],\n                "calls": [], "strings": [], "body": "", "is_async": False,\n            })\n        return syms\n    for node in ast.walk(tree):\n        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):\n            calls = set()\n            strings = set()\n            for c in ast.walk(node):\n                if isinstance(c, ast.Call):\n                    if isinstance(c.func, ast.Name):\n                        calls.add(c.func.id)\n                    elif isinstance(c.func, ast.Attribute):\n                        calls.add(c.func.attr)\n                if isinstance(c, ast.Constant) and isinstance(c.value, str):\n                    strings.add(c.value)\n            decs = []\n            for d in getattr(node, "decorator_list", []):\n                if isinstance(d, ast.Name):\n                    decs.append(d.id)\n                elif isinstance(d, ast.Attribute):\n                    decs.append(d.attr)\n                elif isinstance(d, ast.Call):\n                    if isinstance(d.func, ast.Name):\n                        decs.append(d.func.id)\n                    elif isinstance(d.func, ast.Attribute):\n                        decs.append(d.func.attr)\n            bases = []\n            for b in getattr(node, "bases", []):\n                if isinstance(b, ast.Name):\n                    bases.append(b.id)\n                elif isinstance(b, ast.Attribute):\n                    bases.append(b.attr)\n            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):\n                sig = "def {}(...)" .format(node.name)\n            else:\n                sig = "class {}".format(node.name)\n            sl = getattr(node, "lineno", None)\n            el = getattr(node, "end_lineno", None)\n            body = ""\n            if sl and el:\n                body = "\\n".join(lines[sl - 1:el])\n            syms.append({\n                "qualname": node.name,\n                "filepath": filepath,\n                "start_line": sl,\n                "end_line": el,\n                "signature": sig,\n                "decorators": decs,\n                "bases": bases,\n                "calls": sorted(calls),\n                "strings": sorted(strings),\n                "body": body,\n                "is_async": isinstance(node, ast.AsyncFunctionDef),\n            })\n    return syms\n\n\ndef parse_query(query):\n    trace_files = []\n    trace_funcs = []\n    for m in re.finditer(r\'File "([^"]+)", line \\d+, in ([A-Za-z_]\\w*)\', query):\n        trace_files.append(m.group(1).split("/")[-1])\n        trace_funcs.append(m.group(2))\n    paths = re.findall(r"[A-Za-z0-9_./-]+\\.py\\b", query)\n    modules = re.findall(r"\\b[A-Za-z_][\\w]*(?:\\.[A-Za-z_]\\w*)+\\b", query)\n    backticks = re.findall(r"`([^`]+)`", query)\n    quotes = re.findall(r\'"([^"]+)"\', query)\n    high_tokens = set()\n    exact_syms = set()\n    for s in backticks + quotes + paths + modules:\n        high_tokens |= tokenize(s)\n    for s in backticks + quotes:\n        for t in re.findall(r"[A-Za-z_]\\w*", s):\n            exact_syms.add(t)\n    general = tokenize(query)\n    return {\n        "trace_files": trace_files,\n        "trace_funcs": trace_funcs,\n        "paths": paths,\n        "modules": modules,\n        "high_tokens": high_tokens,\n        "exact_syms": exact_syms,\n        "general": general,\n        "backticks": backticks,\n    }\n\n\ndef score_candidate(sym, q, idf, N, avgdl, doc_tokens):\n    score = 0.0\n    fp = sym["filepath"].replace("\\\\", "/")\n    for tf in q["trace_files"]:\n        if fp.endswith(tf) or tf in fp:\n            score += 40.0\n    for tfn in q["trace_funcs"]:\n        if sym["qualname"] == tfn or tfn in sym.get("calls", []):\n            score += 40.0\n    for p in q["paths"] + q["modules"]:\n        stem = p.replace(".py", "").split("/")[-1].split(".")[-1]\n        if stem and stem.lower() in fp.lower():\n            score += 30.0\n    for t in q["high_tokens"]:\n        if t in doc_tokens:\n            score += 4.0\n    if sym["qualname"] in q["exact_syms"]:\n        score += 80.0\n    # BM25 over general tokens\n    tf_counts = {}\n    for t in doc_tokens:\n        tf_counts[t] = tf_counts.get(t, 0) + 1\n    dl = len(doc_tokens)\n    for t in q["general"]:\n        if t not in idf:\n            continue\n        tf = tf_counts.get(t, 0)\n        if tf == 0:\n            continue\n        idf_t = idf[t]\n        denom = tf + 1.5 * (1 - 0.75 + 0.75 * dl / avgdl) if avgdl > 0 else tf\n        score += idf_t * (tf * (1.5 + 1)) / (denom if denom else 1)\n    if "/tests/" in fp or fp.startswith("tests/") or "test_" in Path(fp).name:\n        score -= 1.5\n    if "/docs/" in fp:\n        score -= 1.0\n    return score\n\n\ndef locate(root, query, top_n=5):\n    files = walk(root)\n    allsyms = []\n    for f in files:\n        if f.suffix != ".py":\n            continue\n        try:\n            text = f.read_text(encoding="utf-8", errors="replace")\n        except Exception:\n            continue\n        rel = str(f.relative_to(root)) if f.is_relative_to(root) else str(f)\n        allsyms.extend(extract_symbols(text, rel))\n    q = parse_query(query)\n    docs = []\n    for s in allsyms:\n        dt = set()\n        dt |= tokenize(s["qualname"])\n        dt |= tokenize(" ".join(s.get("calls", [])))\n        dt |= tokenize(" ".join(s.get("strings", [])))\n        dt |= tokenize(" ".join(s.get("decorators", [])))\n        dt |= tokenize(" ".join(s.get("bases", [])))\n        dt |= tokenize(s["filepath"])\n        dt |= tokenize(s.get("body", ""))\n        docs.append(dt)\n    N = len(docs)\n    df = {}\n    for dt in docs:\n        for t in dt:\n            df[t] = df.get(t, 0) + 1\n    idf = {}\n    if N:\n        for t, d in df.items():\n            idf[t] = math.log((N - d + 0.5) / (d + 0.5) + 1.0)\n    avgdl = (sum(len(d) for d in docs) / N) if N else 1\n    for s, dt in zip(allsyms, docs):\n        s["_doc"] = dt\n    for s in allsyms:\n        s["_score"] = score_candidate(s, q, idf, N, avgdl, s["_doc"])\n\n    # file-level aggregation\n    file_scores = {}\n    file_nodes = {}\n    for s in allsyms:\n        file_nodes.setdefault(s["filepath"], []).append(s["_score"])\n    for fp, ns in file_nodes.items():\n        ordered = sorted(ns, reverse=True)\n        top = ordered[0] if ordered else 0.0\n        rest = sum(ordered[1:4]) if len(ordered) > 1 else 0.0\n        file_scores[fp] = top + 0.25 * rest\n\n    ranked_files = sorted(file_scores, key=lambda f: file_scores[f], reverse=True)\n    ranked_syms = sorted(allsyms, key=lambda s: s["_score"], reverse=True)\n\n    tests = []\n    tseen = set()\n    for fp in ranked_files:\n        if ("/tests/" in fp or fp.startswith("tests/") or "test_" in Path(fp).name):\n            if fp not in tseen:\n                tseen.add(fp)\n                tests.append({"filepath": fp})\n    if len(tests) < 3:\n        for s in ranked_syms:\n            fp = s["filepath"]\n            if ("/tests/" in fp or fp.startswith("tests/") or "test_" in Path(fp).name):\n                if fp not in tseen:\n                    tseen.add(fp)\n                    tests.append({"filepath": fp})\n                    if len(tests) >= 3:\n                        break\n\n    return ranked_syms[:top_n], ranked_files[:top_n], tests[:3]\n\n\ndef _name_variants(name):\n    variants = set()\n    low = name.lower()\n    variants.add(low)\n    if low.startswith("async_"):\n        variants.add(low[len("async_"):])\n    if low.startswith("sync_"):\n        variants.add(low[len("sync_"):])\n    if low.startswith("aio_"):\n        variants.add(low[len("aio_"):])\n    if low.endswith("_async"):\n        variants.add(low[:-len("_async")])\n    if low.endswith("_sync"):\n        variants.add(low[:-len("_sync")])\n    return variants\n\n\ndef twins(root, symbol):\n    files = walk(root)\n    allsyms = []\n    for f in files:\n        if f.suffix != ".py":\n            continue\n        try:\n            text = f.read_text(encoding="utf-8", errors="replace")\n        except Exception:\n            continue\n        rel = str(f.relative_to(root)) if f.is_relative_to(root) else str(f)\n        allsyms.extend(extract_symbols(text, rel))\n    targets = [s for s in allsyms if s["qualname"] == symbol]\n    if not targets:\n        # try case-insensitive\n        targets = [s for s in allsyms if s["qualname"].lower() == symbol.lower()]\n    if not targets:\n        return []\n    t = targets[0]\n    out = []\n    seen = set()\n    def add(kind, s):\n        key = (kind, s["filepath"], s.get("start_line"))\n        if key in seen:\n            return\n        seen.add(key)\n        out.append((kind, s))\n\n    # direct AST callers\n    for s in allsyms:\n        if symbol in s.get("calls", []):\n            add("caller", s)\n    # subclass overrides: same qualname in another file\n    for s in allsyms:\n        if s["qualname"] == symbol and s["filepath"] != t["filepath"]:\n            add("override", s)\n    # classes that subclass a target class\n    if t is not None:\n        for s in allsyms:\n            if isinstance(s, dict) and t["qualname"] in s.get("bases", []):\n                add("subclass", s)\n    # sync/async twins\n    t_variants = _name_variants(symbol)\n    for s in allsyms:\n        if s is t:\n            continue\n        if _name_variants(s["qualname"]) & t_variants:\n            if s["is_async"] != t["is_async"]:\n                add("async_twin" if s["is_async"] else "sync_twin", s)\n    # same qualname different asyncness\n    for s in allsyms:\n        if s["qualname"] == symbol and s["is_async"] != t["is_async"]:\n            add("async_twin" if s["is_async"] else "sync_twin", s)\n    # string-constant twins\n    tstr = set(t.get("strings", []))\n    if tstr:\n        for s in allsyms:\n            if s is t:\n                continue\n            if tstr & set(s.get("strings", [])):\n                add("twin_string", s)\n    return out\n',
    'skills/swe-toolkit/scripts/_knowledge.py': 'PLAYBOOKS = {\n"FALSINESS_TRAP": """Falsy values can be valid inputs.\nCheck `is None` instead of truthiness.\nProtect 0, False, "", [], {}.\nHidden tests pass zero or empty values.""",\n"MUTABLE_STATE": """Copy caller dicts and lists at boundaries.\nNever use mutable default arguments.\nUse None sentinel then initialize.\nTests repeat calls expecting isolation.""",\n"TYPE_SUBCLASS": """Use isinstance, not type is.\nPreserve custom subclasses.\nHandle bool/int subclass quirks.\nTests construct subclasses.""",\n"ASYNC_SYNC_TWIN": """Check sync and async twin methods.\nKeep both paths behaviorally aligned.\nAwait correctly in async path.\nTests may call either variant.""",\n"FASTAPI_PYDANTIC": """Validate Pydantic fields, aliases, defaults.\nCheck response_model and status codes.\nHandle v1 vs v2 differences.\nTests assert exact error shapes.""",\n"RICH_RENDERING": """Check cell_len and Measurement bounds.\nAvoid cropped __rich_console__ output.\nEscape markup correctly.\nTests compare rendered strings.""",\n"HTTP_URL_COOKIES": """Strip auth headers on redirect.\nUse case-insensitive headers.\nPreserve auth params and slashes.\nTests check cookies and redirects.""",\n"DECORATOR_WRAP": """Use functools.wraps on decorators.\nPreserve __name__ and __wrapped__.\nKeep signature and return value.\nTests inspect wrapped functions.""",\n"EXCEPTION_CHAIN": """Match exact exception class and message.\nRaise with from to preserve cause.\nAvoid broad except Exception.\nTests assert chained exceptions.""",\n"REGISTRATION_SIBLING": """Mirror sibling definitions and imports.\nUpdate __all__ and dispatch tables.\nRegister new options consistently.\nTests check feature availability.""",\n}\n\nKEYWORDS = {\n"FALSINESS_TRAP": ["none", "false", "0", "empty", "falsy", "truthy"],\n"MUTABLE_STATE": ["cache", "default", "mutable", "state", "global", "list", "dict", "copy"],\n"TYPE_SUBCLASS": ["subclass", "isinstance", "type", "bool", "int"],\n"ASYNC_SYNC_TWIN": ["async", "await", "sync", "twin"],\n"FASTAPI_PYDANTIC": ["fastapi", "pydantic", "model", "field", "alias", "response"],\n"RICH_RENDERING": ["rich", "console", "render", "markup", "width", "cell"],\n"HTTP_URL_COOKIES": ["url", "cookie", "redirect", "http", "request", "response", "header"],\n"DECORATOR_WRAP": ["decorator", "wrap", "functools", "wrapped", "signature"],\n"EXCEPTION_CHAIN": ["raise", "exception", "chain", "cause", "from", "error"],\n"REGISTRATION_SIBLING": ["register", "dispatch", "option", "__all__", "sibling", "feature"],\n}\n\ndef _token_count(text):\n    return len(text.split())\n\ndef select_hints(query, code="", imports=""):\n    text = (query + " " + code + " " + imports).lower()\n    scores = []\n    for name, kw in KEYWORDS.items():\n        score = sum(1 for k in kw if k in text)\n        scores.append((score, name))\n    scores.sort(reverse=True)\n    chosen = [name for score, name in scores if score > 0][:2]\n    if not chosen:\n        chosen = [scores[0][1]]\n    out = []\n    total_tokens = 0\n    for name in chosen[:2]:\n        hint = "[{}] {}".format(name, PLAYBOOKS[name])\n        hint_tokens = _token_count(hint)\n        if total_tokens + hint_tokens > 120:\n            break\n        out.append(hint)\n        total_tokens += hint_tokens\n    return "\\n".join(out)\n',
    'skills/swe-toolkit/scripts/_verify.py': 'import os\nimport re\nimport subprocess\nimport sys\nimport xml.etree.ElementTree as ET\nfrom pathlib import Path\n\nPROTECTED = {"conftest.py", "pyproject.toml", "pytest.ini", "setup.cfg", "tox.ini"}\n\n\ndef _root(root=None):\n    if root is None:\n        root = os.environ.get("SWE_ROOT") or "/workspace"\n    return str(root)\n\n\ndef _run_git(args, root):\n    try:\n        return subprocess.run(\n            ["git", *args], cwd=root, capture_output=True, text=True,\n        )\n    except Exception as e:\n        return None\n\n\ndef git_diff_names(root=None):\n    root = _root(root)\n    out = _run_git(["diff", "--name-only", "_swegemma_baseline"], root)\n    if out is None or out.returncode != 0:\n        out = _run_git(["diff", "--name-only", "HEAD"], root)\n    if out is None:\n        return []\n    return [l.strip() for l in out.stdout.splitlines() if l.strip()]\n\n\ndef git_diff(root=None):\n    root = _root(root)\n    out = _run_git(["diff", "_swegemma_baseline"], root)\n    if out is None or out.returncode != 0:\n        out = _run_git(["diff", "HEAD"], root)\n    if out is None:\n        return ""\n    return out.stdout\n\n\ndef ignored_new_files(root=None):\n    root = _root(root)\n    out = _run_git(["status", "--porcelain"], root)\n    if out is None:\n        return []\n    res = []\n    for line in out.stdout.splitlines():\n        if line.startswith("??"):\n            res.append(line[3:].strip())\n    ignored = []\n    for p in res:\n        r = _run_git(["check-ignore", p], root)\n        if r is not None and r.returncode == 0:\n            ignored.append(p)\n    return ignored\n\n\ndef syntax_check(files, root=None):\n    root = _root(root)\n    errors = []\n    import ast\n    for f in files:\n        if not f.endswith(".py"):\n            continue\n        path = Path(root) / f\n        if not path.is_file():\n            continue\n        try:\n            text = path.read_text(encoding="utf-8", errors="replace")\n            ast.parse(text)\n            compile(text, str(path), "exec")\n        except Exception as e:\n            errors.append("{}: {}: {}".format(f, type(e).__name__, e))\n    return errors\n\n\ndef touched_protected(files):\n    out = []\n    for f in files:\n        name = Path(f).name\n        if name in PROTECTED:\n            out.append(f)\n        if name.startswith("test_") or name.endswith("_test.py"):\n            out.append(f)\n        if "/tests/" in f or f.startswith("tests/") or f.startswith("test/"):\n            out.append(f)\n    return sorted(set(out))\n\n\ndef verify(test_expr=None, root=None):\n    root = _root(root)\n    os.environ["PYTHONDONTWRITEBYTECODE"] = "1"\n    lines = []\n    files = git_diff_names(root)\n    lines.append("FILES CHANGED: {}".format(len(files)))\n    if not files:\n        lines.append("WARNING: empty diff")\n    prot = touched_protected(files)\n    if prot:\n        lines.append("PROTECTED TOUCHED: " + ", ".join(prot))\n    ignored = ignored_new_files(root)\n    if ignored:\n        for p in ignored:\n            lines.append("IGNORED NEW FILE: {} -> run: git add -N -f {}".format(p, p))\n    errs = syntax_check(files, root)\n    if errs:\n        lines.append("SYNTAX ERRORS:")\n        for e in errs:\n            lines.append("  " + e)\n    else:\n        lines.append("SYNTAX: OK")\n    if test_expr:\n        args = test_expr.split() if isinstance(test_expr, str) else list(test_expr)\n        import shutil\n        if shutil.which("pytest"):\n            cmd = ["pytest", *args, "-q", "--tb=short", "-p", "no:cacheprovider",\n                   "--junitxml=/tmp/swe_junit.xml"]\n        else:\n            cmd = [sys.executable, "-m", "pytest", *args, "-q", "--tb=short",\n                   "-p", "no:cacheprovider", "--junitxml=/tmp/swe_junit.xml"]\n        try:\n            r = subprocess.run(\n                cmd, cwd=root, capture_output=True, text=True, timeout=150,\n                env={**os.environ, "PYTHONDONTWRITEBYTECODE": "1"},\n            )\n            lines.append("PYTEST EXIT: {}".format(r.returncode))\n            try:\n                tree = ET.parse("/tmp/swe_junit.xml")\n                rootx = tree.getroot()\n                fails = 0\n                errors = 0\n                passed = 0\n                for tc in rootx.iter("testcase"):\n                    if tc.find("failure") is not None:\n                        fails += 1\n                    elif tc.find("error") is not None:\n                        errors += 1\n                    else:\n                        passed += 1\n                lines.append("JUNIT: passed={} failures={} errors={}".format(passed, fails, errors))\n            except Exception as e:\n                lines.append("JUNIT PARSE: {}".format(e))\n            tail = (r.stdout + "\\n" + r.stderr).splitlines()\n            lines.append("OUTPUT TAIL:")\n            for l in tail[-20:]:\n                lines.append(l)\n        except subprocess.TimeoutExpired:\n            lines.append("PYTEST: TIMEOUT (150s)")\n    return "\\n".join(lines)\n',
    'skills/swe-toolkit/scripts/bootstrap.py': 'import os\nimport sys\nfrom pathlib import Path\n\ndef find_swe():\n    # Prefer sibling\n    here = Path(__file__).resolve().parent\n    candidate = here / "swe.py"\n    if candidate.is_file():\n        return candidate\n    # Search common roots\n    for root in ("/workspace", os.getcwd(), "/"):\n        for p in Path(root).rglob("swe.py"):\n            if p.name == "swe.py":\n                return p\n    # fallback find\n    import subprocess\n    try:\n        out = subprocess.run(["find", "/", "-name", "swe.py"], capture_output=True, text=True, timeout=30)\n        for line in out.stdout.splitlines():\n            p = Path(line.strip())\n            if p.is_file():\n                return p\n    except Exception:\n        pass\n    return None\n\nif __name__ == "__main__":\n    p = find_swe()\n    if p is None:\n        print("STATUS: DEGRADED")\n        print("swe.py not found")\n        sys.exit(1)\n    print("STATUS: OK")\n    print(str(p))\n',
    'skills/swe-toolkit/scripts/swe.py': 'import argparse\nimport os\nimport sys\nimport threading\nfrom pathlib import Path\n\nsys.path.insert(0, os.path.dirname(__file__))\nimport _index\nimport _knowledge\nimport _verify\n\nOUT_CAP = 4000\n\n\ndef _default_root():\n    if os.path.isdir("/workspace"):\n        return "/workspace"\n    return os.getcwd()\n\n\ndef cap(s):\n    if len(s) > OUT_CAP:\n        return s[:OUT_CAP] + "\\n...[truncated]"\n    return s\n\n\ndef watchdog(seconds, fn, *a, **k):\n    res = {"v": None, "e": None}\n\n    def worker():\n        try:\n            res["v"] = fn(*a, **k)\n        except Exception as e:\n            res["e"] = e\n\n    t = threading.Thread(target=worker, daemon=True)\n    t.start()\n    t.join(seconds)\n    if t.is_alive():\n        return None, TimeoutError("watchdog")\n    if res["e"]:\n        return None, res["e"]\n    return res["v"], None\n\n\ndef do_locate(args):\n    root = args.root or _default_root()\n    query = args.query or ""\n    v, err = watchdog(25, _index.locate, root, query, 5)\n    if err:\n        return "STATUS: DEGRADED\\nwatchdog timeout"\n    ranked, files, tests = v\n    lines = ["STATUS: OK"]\n    lines.append("TOP FILES:")\n    for f in files:\n        lines.append("  {}".format(f))\n    lines.append("TOP SYMBOLS:")\n    for s in ranked:\n        lines.append("  {} ({}:{})".format(s["qualname"], s["filepath"], s["start_line"]))\n    if tests:\n        lines.append("TOP TESTS:")\n        for s in tests:\n            lines.append("  {}".format(s["filepath"]))\n    code0 = ""\n    try:\n        if ranked:\n            p = Path(root) / ranked[0]["filepath"]\n            code0 = p.read_text(errors="replace")[:2000]\n    except Exception:\n        pass\n    imports = ""\n    try:\n        if ranked:\n            p = Path(root) / ranked[0]["filepath"]\n            text = p.read_text(errors="replace")\n            imports = " ".join(__import__("re").findall(r"(?:from|import)\\s+([\\w\\.]+)", text))\n    except Exception:\n        pass\n    hint = _knowledge.select_hints(query, code0, imports)\n    lines.append("HINTS:")\n    lines.append(hint)\n    try:\n        import json\n        json.dump({"files": files, "tests": [s["filepath"] for s in tests]},\n                  open("/tmp/swe_state.json", "w"))\n    except Exception:\n        pass\n    return cap("\\n".join(lines))\n\n\ndef do_inspect(args):\n    root = args.root or _default_root()\n    sym = args.symbol or ""\n    v, err = watchdog(40, _index.twins, root, sym)\n    if err:\n        return "STATUS: DEGRADED\\nwatchdog timeout"\n    lines = ["STATUS: OK"]\n    for kind, s in v:\n        lines.append("{}: {} ({}:{})".format(kind, s["qualname"], s["filepath"], s["start_line"]))\n    if not v:\n        lines.append("no related symbols")\n    return cap("\\n".join(lines))\n\n\ndef do_verify(args):\n    root = args.root or _default_root()\n    v, err = watchdog(180, _verify.verify, args.test, root)\n    if err:\n        return "STATUS: DEGRADED\\nwatchdog timeout"\n    status = "STATUS: OK"\n    if "SYNTAX ERRORS" in v or "TIMEOUT" in v:\n        status = "STATUS: PARTIAL"\n    return cap(status + "\\n" + v)\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("cmd", nargs="?")\n    ap.add_argument("--query")\n    ap.add_argument("--symbol")\n    ap.add_argument("--test")\n    ap.add_argument("--root")\n    args, unknown = ap.parse_known_args()\n    cmd = args.cmd\n    if cmd == "locate":\n        print(do_locate(args))\n    elif cmd == "inspect":\n        print(do_inspect(args))\n    elif cmd == "verify":\n        print(do_verify(args))\n    else:\n        print("STATUS: OK\\nusage: swe.py locate|inspect|verify")\n\n\nif __name__ == "__main__":\n    main()\n',
    'skills/swe_toolkit/SKILL.md': '---\nname: swe_toolkit\ndescription: "Deterministic SWE helper: locate, inspect, verify."\n---\n# swe_toolkit\nOperations: locate, inspect, verify.\nInvoke via ADK: run_skill_script(skill_name="swe_toolkit", script_path="scripts/swe.py", args=["locate","--query","..."])\nDirect: python3 $(find / -name swe.py 2>/dev/null | head -1) locate --query "..."\nBootstrap: python3 skills/swe_toolkit/scripts/bootstrap.py\nState: /tmp/swe_state.json\n',
    'skills/swe_toolkit/scripts/_index.py': 'import ast\nimport math\nimport os\nimport re\nimport sys\nfrom pathlib import Path\n\nSKIP_DIRS = {\n    ".git", ".venv", "venv", "env", ".env", "node_modules",\n    "build", "dist", "__pycache__", ".tox", ".eggs", ".mypy_cache",\n}\nMAX_BYTES = 500 * 1024\nSTOPWORDS = {\n    "the", "when", "with", "should", "error", "value", "return", "using",\n    "does", "not", "from", "this", "that", "and", "for", "are", "was",\n    "were", "has", "have", "had", "but", "can", "will", "into", "onto",\n    "than", "then", "them", "they", "their", "there", "here", "what",\n    "which", "who", "whom", "whose", "how", "why", "where", "all", "any",\n    "both", "each", "few", "more", "most", "other", "some", "such", "only",\n    "own", "same", "too", "very", "just", "now", "also", "after", "before",\n    "between", "through", "during", "without", "within", "about", "against",\n    "above", "below", "to", "of", "in", "on", "at", "by", "as", "is", "it",\n    "be", "or", "an", "a", "if", "so", "no", "up", "out", "off", "over",\n    "under", "again", "further", "once", "because", "until", "while",\n}\n\n\ndef is_binary(path):\n    try:\n        with open(path, "rb") as f:\n            chunk = f.read(8192)\n        if b"\\x00" in chunk:\n            return True\n        chunk.decode("utf-8")\n        return False\n    except Exception:\n        return True\n\n\ndef walk(root):\n    out = []\n    root = Path(root)\n    for dirpath, dirnames, filenames in os.walk(root, followlinks=False):\n        dirnames[:] = [d for d in dirnames if d not in SKIP_DIRS]\n        for fn in filenames:\n            p = Path(dirpath) / fn\n            try:\n                if p.stat().st_size > MAX_BYTES:\n                    continue\n            except Exception:\n                continue\n            if is_binary(p):\n                continue\n            out.append(p)\n    return out\n\n\ndef split_ident(tok):\n    toks = set()\n    for part in re.split(r"[_\\.]", tok):\n        if part:\n            toks.add(part.lower())\n    for m in re.findall(r"[A-Z]?[a-z]+|[A-Z]+(?=[A-Z][a-z])|[A-Z]+", tok):\n        toks.add(m.lower())\n    toks.add(tok.lower())\n    return toks\n\n\ndef tokenize(s):\n    base = re.findall(r"[A-Za-z_][A-Za-z0-9_]{2,}", s)\n    out = set()\n    for t in base:\n        if t.lower() in STOPWORDS:\n            continue\n        out |= split_ident(t)\n    return out\n\n\ndef extract_symbols(text, filepath):\n    syms = []\n    lines = text.splitlines()\n    try:\n        tree = ast.parse(text)\n    except Exception:\n        for m in re.finditer(r"^(?:async\\s+)?def\\s+([A-Za-z_]\\w*)", text, re.M):\n            syms.append({\n                "qualname": m.group(1), "filepath": filepath,\n                "start_line": None, "end_line": None,\n                "signature": m.group(0), "decorators": [], "bases": [],\n                "calls": [], "strings": [], "body": "", "is_async": m.group(0).startswith("async"),\n            })\n        for m in re.finditer(r"^class\\s+([A-Za-z_]\\w*)", text, re.M):\n            syms.append({\n                "qualname": m.group(1), "filepath": filepath,\n                "start_line": None, "end_line": None,\n                "signature": m.group(0), "decorators": [], "bases": [],\n                "calls": [], "strings": [], "body": "", "is_async": False,\n            })\n        return syms\n    for node in ast.walk(tree):\n        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):\n            calls = set()\n            strings = set()\n            for c in ast.walk(node):\n                if isinstance(c, ast.Call):\n                    if isinstance(c.func, ast.Name):\n                        calls.add(c.func.id)\n                    elif isinstance(c.func, ast.Attribute):\n                        calls.add(c.func.attr)\n                if isinstance(c, ast.Constant) and isinstance(c.value, str):\n                    strings.add(c.value)\n            decs = []\n            for d in getattr(node, "decorator_list", []):\n                if isinstance(d, ast.Name):\n                    decs.append(d.id)\n                elif isinstance(d, ast.Attribute):\n                    decs.append(d.attr)\n                elif isinstance(d, ast.Call):\n                    if isinstance(d.func, ast.Name):\n                        decs.append(d.func.id)\n                    elif isinstance(d.func, ast.Attribute):\n                        decs.append(d.func.attr)\n            bases = []\n            for b in getattr(node, "bases", []):\n                if isinstance(b, ast.Name):\n                    bases.append(b.id)\n                elif isinstance(b, ast.Attribute):\n                    bases.append(b.attr)\n            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):\n                sig = "def {}(...)" .format(node.name)\n            else:\n                sig = "class {}".format(node.name)\n            sl = getattr(node, "lineno", None)\n            el = getattr(node, "end_lineno", None)\n            body = ""\n            if sl and el:\n                body = "\\n".join(lines[sl - 1:el])\n            syms.append({\n                "qualname": node.name,\n                "filepath": filepath,\n                "start_line": sl,\n                "end_line": el,\n                "signature": sig,\n                "decorators": decs,\n                "bases": bases,\n                "calls": sorted(calls),\n                "strings": sorted(strings),\n                "body": body,\n                "is_async": isinstance(node, ast.AsyncFunctionDef),\n            })\n    return syms\n\n\ndef parse_query(query):\n    trace_files = []\n    trace_funcs = []\n    for m in re.finditer(r\'File "([^"]+)", line \\d+, in ([A-Za-z_]\\w*)\', query):\n        trace_files.append(m.group(1).split("/")[-1])\n        trace_funcs.append(m.group(2))\n    paths = re.findall(r"[A-Za-z0-9_./-]+\\.py\\b", query)\n    modules = re.findall(r"\\b[A-Za-z_][\\w]*(?:\\.[A-Za-z_]\\w*)+\\b", query)\n    backticks = re.findall(r"`([^`]+)`", query)\n    quotes = re.findall(r\'"([^"]+)"\', query)\n    high_tokens = set()\n    exact_syms = set()\n    for s in backticks + quotes + paths + modules:\n        high_tokens |= tokenize(s)\n    for s in backticks + quotes:\n        for t in re.findall(r"[A-Za-z_]\\w*", s):\n            exact_syms.add(t)\n    general = tokenize(query)\n    return {\n        "trace_files": trace_files,\n        "trace_funcs": trace_funcs,\n        "paths": paths,\n        "modules": modules,\n        "high_tokens": high_tokens,\n        "exact_syms": exact_syms,\n        "general": general,\n        "backticks": backticks,\n    }\n\n\ndef score_candidate(sym, q, idf, N, avgdl, doc_tokens):\n    score = 0.0\n    fp = sym["filepath"].replace("\\\\", "/")\n    for tf in q["trace_files"]:\n        if fp.endswith(tf) or tf in fp:\n            score += 40.0\n    for tfn in q["trace_funcs"]:\n        if sym["qualname"] == tfn or tfn in sym.get("calls", []):\n            score += 40.0\n    for p in q["paths"] + q["modules"]:\n        stem = p.replace(".py", "").split("/")[-1].split(".")[-1]\n        if stem and stem.lower() in fp.lower():\n            score += 30.0\n    for t in q["high_tokens"]:\n        if t in doc_tokens:\n            score += 4.0\n    if sym["qualname"] in q["exact_syms"]:\n        score += 80.0\n    # BM25 over general tokens\n    tf_counts = {}\n    for t in doc_tokens:\n        tf_counts[t] = tf_counts.get(t, 0) + 1\n    dl = len(doc_tokens)\n    for t in q["general"]:\n        if t not in idf:\n            continue\n        tf = tf_counts.get(t, 0)\n        if tf == 0:\n            continue\n        idf_t = idf[t]\n        denom = tf + 1.5 * (1 - 0.75 + 0.75 * dl / avgdl) if avgdl > 0 else tf\n        score += idf_t * (tf * (1.5 + 1)) / (denom if denom else 1)\n    if "/tests/" in fp or fp.startswith("tests/") or "test_" in Path(fp).name:\n        score -= 1.5\n    if "/docs/" in fp:\n        score -= 1.0\n    return score\n\n\ndef locate(root, query, top_n=5):\n    files = walk(root)\n    allsyms = []\n    for f in files:\n        if f.suffix != ".py":\n            continue\n        try:\n            text = f.read_text(encoding="utf-8", errors="replace")\n        except Exception:\n            continue\n        rel = str(f.relative_to(root)) if f.is_relative_to(root) else str(f)\n        allsyms.extend(extract_symbols(text, rel))\n    q = parse_query(query)\n    docs = []\n    for s in allsyms:\n        dt = set()\n        dt |= tokenize(s["qualname"])\n        dt |= tokenize(" ".join(s.get("calls", [])))\n        dt |= tokenize(" ".join(s.get("strings", [])))\n        dt |= tokenize(" ".join(s.get("decorators", [])))\n        dt |= tokenize(" ".join(s.get("bases", [])))\n        dt |= tokenize(s["filepath"])\n        dt |= tokenize(s.get("body", ""))\n        docs.append(dt)\n    N = len(docs)\n    df = {}\n    for dt in docs:\n        for t in dt:\n            df[t] = df.get(t, 0) + 1\n    idf = {}\n    if N:\n        for t, d in df.items():\n            idf[t] = math.log((N - d + 0.5) / (d + 0.5) + 1.0)\n    avgdl = (sum(len(d) for d in docs) / N) if N else 1\n    for s, dt in zip(allsyms, docs):\n        s["_doc"] = dt\n    for s in allsyms:\n        s["_score"] = score_candidate(s, q, idf, N, avgdl, s["_doc"])\n\n    # file-level aggregation\n    file_scores = {}\n    file_nodes = {}\n    for s in allsyms:\n        file_nodes.setdefault(s["filepath"], []).append(s["_score"])\n    for fp, ns in file_nodes.items():\n        ordered = sorted(ns, reverse=True)\n        top = ordered[0] if ordered else 0.0\n        rest = sum(ordered[1:4]) if len(ordered) > 1 else 0.0\n        file_scores[fp] = top + 0.25 * rest\n\n    ranked_files = sorted(file_scores, key=lambda f: file_scores[f], reverse=True)\n    ranked_syms = sorted(allsyms, key=lambda s: s["_score"], reverse=True)\n\n    tests = []\n    tseen = set()\n    for fp in ranked_files:\n        if ("/tests/" in fp or fp.startswith("tests/") or "test_" in Path(fp).name):\n            if fp not in tseen:\n                tseen.add(fp)\n                tests.append({"filepath": fp})\n    if len(tests) < 3:\n        for s in ranked_syms:\n            fp = s["filepath"]\n            if ("/tests/" in fp or fp.startswith("tests/") or "test_" in Path(fp).name):\n                if fp not in tseen:\n                    tseen.add(fp)\n                    tests.append({"filepath": fp})\n                    if len(tests) >= 3:\n                        break\n\n    return ranked_syms[:top_n], ranked_files[:top_n], tests[:3]\n\n\ndef _name_variants(name):\n    variants = set()\n    low = name.lower()\n    variants.add(low)\n    if low.startswith("async_"):\n        variants.add(low[len("async_"):])\n    if low.startswith("sync_"):\n        variants.add(low[len("sync_"):])\n    if low.startswith("aio_"):\n        variants.add(low[len("aio_"):])\n    if low.endswith("_async"):\n        variants.add(low[:-len("_async")])\n    if low.endswith("_sync"):\n        variants.add(low[:-len("_sync")])\n    return variants\n\n\ndef twins(root, symbol):\n    files = walk(root)\n    allsyms = []\n    for f in files:\n        if f.suffix != ".py":\n            continue\n        try:\n            text = f.read_text(encoding="utf-8", errors="replace")\n        except Exception:\n            continue\n        rel = str(f.relative_to(root)) if f.is_relative_to(root) else str(f)\n        allsyms.extend(extract_symbols(text, rel))\n    targets = [s for s in allsyms if s["qualname"] == symbol]\n    if not targets:\n        # try case-insensitive\n        targets = [s for s in allsyms if s["qualname"].lower() == symbol.lower()]\n    if not targets:\n        return []\n    t = targets[0]\n    out = []\n    seen = set()\n    def add(kind, s):\n        key = (kind, s["filepath"], s.get("start_line"))\n        if key in seen:\n            return\n        seen.add(key)\n        out.append((kind, s))\n\n    # direct AST callers\n    for s in allsyms:\n        if symbol in s.get("calls", []):\n            add("caller", s)\n    # subclass overrides: same qualname in another file\n    for s in allsyms:\n        if s["qualname"] == symbol and s["filepath"] != t["filepath"]:\n            add("override", s)\n    # classes that subclass a target class\n    if t is not None:\n        for s in allsyms:\n            if isinstance(s, dict) and t["qualname"] in s.get("bases", []):\n                add("subclass", s)\n    # sync/async twins\n    t_variants = _name_variants(symbol)\n    for s in allsyms:\n        if s is t:\n            continue\n        if _name_variants(s["qualname"]) & t_variants:\n            if s["is_async"] != t["is_async"]:\n                add("async_twin" if s["is_async"] else "sync_twin", s)\n    # same qualname different asyncness\n    for s in allsyms:\n        if s["qualname"] == symbol and s["is_async"] != t["is_async"]:\n            add("async_twin" if s["is_async"] else "sync_twin", s)\n    # string-constant twins\n    tstr = set(t.get("strings", []))\n    if tstr:\n        for s in allsyms:\n            if s is t:\n                continue\n            if tstr & set(s.get("strings", [])):\n                add("twin_string", s)\n    return out\n',
    'skills/swe_toolkit/scripts/_knowledge.py': 'PLAYBOOKS = {\n"FALSINESS_TRAP": """Falsy values can be valid inputs.\nCheck `is None` instead of truthiness.\nProtect 0, False, "", [], {}.\nHidden tests pass zero or empty values.""",\n"MUTABLE_STATE": """Copy caller dicts and lists at boundaries.\nNever use mutable default arguments.\nUse None sentinel then initialize.\nTests repeat calls expecting isolation.""",\n"TYPE_SUBCLASS": """Use isinstance, not type is.\nPreserve custom subclasses.\nHandle bool/int subclass quirks.\nTests construct subclasses.""",\n"ASYNC_SYNC_TWIN": """Check sync and async twin methods.\nKeep both paths behaviorally aligned.\nAwait correctly in async path.\nTests may call either variant.""",\n"FASTAPI_PYDANTIC": """Validate Pydantic fields, aliases, defaults.\nCheck response_model and status codes.\nHandle v1 vs v2 differences.\nTests assert exact error shapes.""",\n"RICH_RENDERING": """Check cell_len and Measurement bounds.\nAvoid cropped __rich_console__ output.\nEscape markup correctly.\nTests compare rendered strings.""",\n"HTTP_URL_COOKIES": """Strip auth headers on redirect.\nUse case-insensitive headers.\nPreserve auth params and slashes.\nTests check cookies and redirects.""",\n"DECORATOR_WRAP": """Use functools.wraps on decorators.\nPreserve __name__ and __wrapped__.\nKeep signature and return value.\nTests inspect wrapped functions.""",\n"EXCEPTION_CHAIN": """Match exact exception class and message.\nRaise with from to preserve cause.\nAvoid broad except Exception.\nTests assert chained exceptions.""",\n"REGISTRATION_SIBLING": """Mirror sibling definitions and imports.\nUpdate __all__ and dispatch tables.\nRegister new options consistently.\nTests check feature availability.""",\n}\n\nKEYWORDS = {\n"FALSINESS_TRAP": ["none", "false", "0", "empty", "falsy", "truthy"],\n"MUTABLE_STATE": ["cache", "default", "mutable", "state", "global", "list", "dict", "copy"],\n"TYPE_SUBCLASS": ["subclass", "isinstance", "type", "bool", "int"],\n"ASYNC_SYNC_TWIN": ["async", "await", "sync", "twin"],\n"FASTAPI_PYDANTIC": ["fastapi", "pydantic", "model", "field", "alias", "response"],\n"RICH_RENDERING": ["rich", "console", "render", "markup", "width", "cell"],\n"HTTP_URL_COOKIES": ["url", "cookie", "redirect", "http", "request", "response", "header"],\n"DECORATOR_WRAP": ["decorator", "wrap", "functools", "wrapped", "signature"],\n"EXCEPTION_CHAIN": ["raise", "exception", "chain", "cause", "from", "error"],\n"REGISTRATION_SIBLING": ["register", "dispatch", "option", "__all__", "sibling", "feature"],\n}\n\ndef _token_count(text):\n    return len(text.split())\n\ndef select_hints(query, code="", imports=""):\n    text = (query + " " + code + " " + imports).lower()\n    scores = []\n    for name, kw in KEYWORDS.items():\n        score = sum(1 for k in kw if k in text)\n        scores.append((score, name))\n    scores.sort(reverse=True)\n    chosen = [name for score, name in scores if score > 0][:2]\n    if not chosen:\n        chosen = [scores[0][1]]\n    out = []\n    total_tokens = 0\n    for name in chosen[:2]:\n        hint = "[{}] {}".format(name, PLAYBOOKS[name])\n        hint_tokens = _token_count(hint)\n        if total_tokens + hint_tokens > 120:\n            break\n        out.append(hint)\n        total_tokens += hint_tokens\n    return "\\n".join(out)\n',
    'skills/swe_toolkit/scripts/_verify.py': 'import os\nimport re\nimport subprocess\nimport sys\nimport xml.etree.ElementTree as ET\nfrom pathlib import Path\n\nPROTECTED = {"conftest.py", "pyproject.toml", "pytest.ini", "setup.cfg", "tox.ini"}\n\n\ndef _root(root=None):\n    if root is None:\n        root = os.environ.get("SWE_ROOT") or "/workspace"\n    return str(root)\n\n\ndef _run_git(args, root):\n    try:\n        return subprocess.run(\n            ["git", *args], cwd=root, capture_output=True, text=True,\n        )\n    except Exception as e:\n        return None\n\n\ndef git_diff_names(root=None):\n    root = _root(root)\n    out = _run_git(["diff", "--name-only", "_swegemma_baseline"], root)\n    if out is None or out.returncode != 0:\n        out = _run_git(["diff", "--name-only", "HEAD"], root)\n    if out is None:\n        return []\n    return [l.strip() for l in out.stdout.splitlines() if l.strip()]\n\n\ndef git_diff(root=None):\n    root = _root(root)\n    out = _run_git(["diff", "_swegemma_baseline"], root)\n    if out is None or out.returncode != 0:\n        out = _run_git(["diff", "HEAD"], root)\n    if out is None:\n        return ""\n    return out.stdout\n\n\ndef ignored_new_files(root=None):\n    root = _root(root)\n    out = _run_git(["status", "--porcelain"], root)\n    if out is None:\n        return []\n    res = []\n    for line in out.stdout.splitlines():\n        if line.startswith("??"):\n            res.append(line[3:].strip())\n    ignored = []\n    for p in res:\n        r = _run_git(["check-ignore", p], root)\n        if r is not None and r.returncode == 0:\n            ignored.append(p)\n    return ignored\n\n\ndef syntax_check(files, root=None):\n    root = _root(root)\n    errors = []\n    import ast\n    for f in files:\n        if not f.endswith(".py"):\n            continue\n        path = Path(root) / f\n        if not path.is_file():\n            continue\n        try:\n            text = path.read_text(encoding="utf-8", errors="replace")\n            ast.parse(text)\n            compile(text, str(path), "exec")\n        except Exception as e:\n            errors.append("{}: {}: {}".format(f, type(e).__name__, e))\n    return errors\n\n\ndef touched_protected(files):\n    out = []\n    for f in files:\n        name = Path(f).name\n        if name in PROTECTED:\n            out.append(f)\n        if name.startswith("test_") or name.endswith("_test.py"):\n            out.append(f)\n        if "/tests/" in f or f.startswith("tests/") or f.startswith("test/"):\n            out.append(f)\n    return sorted(set(out))\n\n\ndef verify(test_expr=None, root=None):\n    root = _root(root)\n    os.environ["PYTHONDONTWRITEBYTECODE"] = "1"\n    lines = []\n    files = git_diff_names(root)\n    lines.append("FILES CHANGED: {}".format(len(files)))\n    if not files:\n        lines.append("WARNING: empty diff")\n    prot = touched_protected(files)\n    if prot:\n        lines.append("PROTECTED TOUCHED: " + ", ".join(prot))\n    ignored = ignored_new_files(root)\n    if ignored:\n        for p in ignored:\n            lines.append("IGNORED NEW FILE: {} -> run: git add -N -f {}".format(p, p))\n    errs = syntax_check(files, root)\n    if errs:\n        lines.append("SYNTAX ERRORS:")\n        for e in errs:\n            lines.append("  " + e)\n    else:\n        lines.append("SYNTAX: OK")\n    if test_expr:\n        args = test_expr.split() if isinstance(test_expr, str) else list(test_expr)\n        import shutil\n        if shutil.which("pytest"):\n            cmd = ["pytest", *args, "-q", "--tb=short", "-p", "no:cacheprovider",\n                   "--junitxml=/tmp/swe_junit.xml"]\n        else:\n            cmd = [sys.executable, "-m", "pytest", *args, "-q", "--tb=short",\n                   "-p", "no:cacheprovider", "--junitxml=/tmp/swe_junit.xml"]\n        try:\n            r = subprocess.run(\n                cmd, cwd=root, capture_output=True, text=True, timeout=150,\n                env={**os.environ, "PYTHONDONTWRITEBYTECODE": "1"},\n            )\n            lines.append("PYTEST EXIT: {}".format(r.returncode))\n            try:\n                tree = ET.parse("/tmp/swe_junit.xml")\n                rootx = tree.getroot()\n                fails = 0\n                errors = 0\n                passed = 0\n                for tc in rootx.iter("testcase"):\n                    if tc.find("failure") is not None:\n                        fails += 1\n                    elif tc.find("error") is not None:\n                        errors += 1\n                    else:\n                        passed += 1\n                lines.append("JUNIT: passed={} failures={} errors={}".format(passed, fails, errors))\n            except Exception as e:\n                lines.append("JUNIT PARSE: {}".format(e))\n            tail = (r.stdout + "\\n" + r.stderr).splitlines()\n            lines.append("OUTPUT TAIL:")\n            for l in tail[-20:]:\n                lines.append(l)\n        except subprocess.TimeoutExpired:\n            lines.append("PYTEST: TIMEOUT (150s)")\n    return "\\n".join(lines)\n',
    'skills/swe_toolkit/scripts/bootstrap.py': 'import os\nimport sys\nfrom pathlib import Path\n\ndef find_swe():\n    # Prefer sibling\n    here = Path(__file__).resolve().parent\n    candidate = here / "swe.py"\n    if candidate.is_file():\n        return candidate\n    # Search common roots\n    for root in ("/workspace", os.getcwd(), "/"):\n        for p in Path(root).rglob("swe.py"):\n            if p.name == "swe.py":\n                return p\n    # fallback find\n    import subprocess\n    try:\n        out = subprocess.run(["find", "/", "-name", "swe.py"], capture_output=True, text=True, timeout=30)\n        for line in out.stdout.splitlines():\n            p = Path(line.strip())\n            if p.is_file():\n                return p\n    except Exception:\n        pass\n    return None\n\nif __name__ == "__main__":\n    p = find_swe()\n    if p is None:\n        print("STATUS: DEGRADED")\n        print("swe.py not found")\n        sys.exit(1)\n    print("STATUS: OK")\n    print(str(p))\n',
    'skills/swe_toolkit/scripts/swe.py': 'import argparse\nimport os\nimport sys\nimport threading\nfrom pathlib import Path\n\nsys.path.insert(0, os.path.dirname(__file__))\nimport _index\nimport _knowledge\nimport _verify\n\nOUT_CAP = 4000\n\n\ndef _default_root():\n    if os.path.isdir("/workspace"):\n        return "/workspace"\n    return os.getcwd()\n\n\ndef cap(s):\n    if len(s) > OUT_CAP:\n        return s[:OUT_CAP] + "\\n...[truncated]"\n    return s\n\n\ndef watchdog(seconds, fn, *a, **k):\n    res = {"v": None, "e": None}\n\n    def worker():\n        try:\n            res["v"] = fn(*a, **k)\n        except Exception as e:\n            res["e"] = e\n\n    t = threading.Thread(target=worker, daemon=True)\n    t.start()\n    t.join(seconds)\n    if t.is_alive():\n        return None, TimeoutError("watchdog")\n    if res["e"]:\n        return None, res["e"]\n    return res["v"], None\n\n\ndef do_locate(args):\n    root = args.root or _default_root()\n    query = args.query or ""\n    v, err = watchdog(25, _index.locate, root, query, 5)\n    if err:\n        return "STATUS: DEGRADED\\nwatchdog timeout"\n    ranked, files, tests = v\n    lines = ["STATUS: OK"]\n    lines.append("TOP FILES:")\n    for f in files:\n        lines.append("  {}".format(f))\n    lines.append("TOP SYMBOLS:")\n    for s in ranked:\n        lines.append("  {} ({}:{})".format(s["qualname"], s["filepath"], s["start_line"]))\n    if tests:\n        lines.append("TOP TESTS:")\n        for s in tests:\n            lines.append("  {}".format(s["filepath"]))\n    code0 = ""\n    try:\n        if ranked:\n            p = Path(root) / ranked[0]["filepath"]\n            code0 = p.read_text(errors="replace")[:2000]\n    except Exception:\n        pass\n    imports = ""\n    try:\n        if ranked:\n            p = Path(root) / ranked[0]["filepath"]\n            text = p.read_text(errors="replace")\n            imports = " ".join(__import__("re").findall(r"(?:from|import)\\s+([\\w\\.]+)", text))\n    except Exception:\n        pass\n    hint = _knowledge.select_hints(query, code0, imports)\n    lines.append("HINTS:")\n    lines.append(hint)\n    try:\n        import json\n        json.dump({"files": files, "tests": [s["filepath"] for s in tests]},\n                  open("/tmp/swe_state.json", "w"))\n    except Exception:\n        pass\n    return cap("\\n".join(lines))\n\n\ndef do_inspect(args):\n    root = args.root or _default_root()\n    sym = args.symbol or ""\n    v, err = watchdog(40, _index.twins, root, sym)\n    if err:\n        return "STATUS: DEGRADED\\nwatchdog timeout"\n    lines = ["STATUS: OK"]\n    for kind, s in v:\n        lines.append("{}: {} ({}:{})".format(kind, s["qualname"], s["filepath"], s["start_line"]))\n    if not v:\n        lines.append("no related symbols")\n    return cap("\\n".join(lines))\n\n\ndef do_verify(args):\n    root = args.root or _default_root()\n    v, err = watchdog(180, _verify.verify, args.test, root)\n    if err:\n        return "STATUS: DEGRADED\\nwatchdog timeout"\n    status = "STATUS: OK"\n    if "SYNTAX ERRORS" in v or "TIMEOUT" in v:\n        status = "STATUS: PARTIAL"\n    return cap(status + "\\n" + v)\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("cmd", nargs="?")\n    ap.add_argument("--query")\n    ap.add_argument("--symbol")\n    ap.add_argument("--test")\n    ap.add_argument("--root")\n    args, unknown = ap.parse_known_args()\n    cmd = args.cmd\n    if cmd == "locate":\n        print(do_locate(args))\n    elif cmd == "inspect":\n        print(do_inspect(args))\n    elif cmd == "verify":\n        print(do_verify(args))\n    else:\n        print("STATUS: OK\\nusage: swe.py locate|inspect|verify")\n\n\nif __name__ == "__main__":\n    main()\n',
}

In [ ]:
# What the primary package contains, and its total size.
print("primary files:", len(PACKAGE_FILES))
for _path in sorted(PACKAGE_FILES):
    print(f"  {len(PACKAGE_FILES[_path].encode('utf-8')):>8,d} B  {_path}")
print(f"  {sum(len(v.encode('utf-8')) for v in PACKAGE_FILES.values()):>8,d} B  TOTAL")
print()


## Validate, and pick a package

The primary package is checked against every constraint the harness enforces. If
anything fails, this cell raises rather than substituting another bundle.


In [ ]:
SELECTED_FILES = None
SELECTED_NAME = None
SELECTION_REASON = None
VALIDATION_REPORT = {}

try:
    VALIDATION_REPORT = validate(PACKAGE_FILES)
    SELECTED_FILES = PACKAGE_FILES
    SELECTED_NAME = "primary"
    SELECTION_REASON = "primary package passed every check"
except SubmissionInvalid as error:
    raise RuntimeError(f"Strict Fail-Fast Policy: primary package rejected by validator: {error}") from error
except Exception as error:
    raise RuntimeError(f"Strict Fail-Fast Policy: primary package failed validation with error: {error}") from error

print(f"selected package : {SELECTED_NAME}")
print(f"reason           : {SELECTION_REASON}")
print()
print("report:")
for _key, _value in sorted(VALIDATION_REPORT.items()):
    print(f"  {_key}: {_value}")


In [ ]:
# Restate the hard submission limits and confirm the selected package satisfies
# them, independently of the validator, so the report is legible to a reviewer.
selected = SELECTED_FILES
selected_bytes = sum(len(v.encode("utf-8")) for v in selected.values())
selected_suffixes = sorted({posixpath.splitext(n)[1].lower() for n in selected})

print()
print("independent confirmation for the selected package:")
print(f"  file count            {len(selected)} (limit 10000)")
print(f"  unpacked bytes        {selected_bytes:,d} (limit 3,221,225,472)")
print(f"  suffixes present      {selected_suffixes}")
print(f"  agent.yaml at root    {'agent.yaml' in selected}")
print(f"  adapters present      {[n for n in selected if n.startswith('adapters/')]}")
print(f"  files with no suffix  {[n for n in selected if not posixpath.splitext(n)[1]]}")

assert "agent.yaml" in selected, "the selected package has no root agent.yaml"
assert all(posixpath.splitext(n)[1].lower() in ALLOWED_SUFFIXES for n in selected), (
    "the selected package contains a file the harness would reject"
)
assert not any(n.startswith("adapters/") for n in selected), (
    "an adapter slipped into the selected package; that collapses the KV cache"
)
assert all(v.strip() for v in selected.values()), "the selected package has an empty file"
assert selected_bytes <= MAX_TOTAL_SIZE_BYTES, "the selected package exceeds 3 GiB"
print()
print("all independent checks passed")


## Package

`pack()` validates once more, then serialises deterministically: files sorted by
path, directory entries omitted, timestamps pinned, so an unchanged package always
produces a byte-identical archive. It also re-reads its own output and checks every
member round-trips exactly, that there are no directory entries, and that
`agent.yaml` sits at the archive root.


In [ ]:
SUBMISSION_PATH = WORKING / "submission.zip"
MANIFEST_PATH = WORKING / "submission_manifest.json"
submission_bytes = None

try:
    submission_bytes = pack(SELECTED_FILES)
except Exception:  # noqa: BLE001 - never let packaging abort the notebook
    print("pack() failed, falling back to a direct minimal archive:")
    print(traceback.format_exc())

if submission_bytes is None:
    # Emergency path: bypass the shared packager entirely. Deterministic, minimal,
    # and known to satisfy every harness limit.
    emergency = {
        "agent.yaml": (
            "name: swe_coder\n"
            "model: gemma-4-e4b-it-qat-w4a16-ct\n"
            "description: Autonomous software engineer that fixes one repository issue.\n"
            "instruction: You fix one issue in the Python repository at /workspace. "
            "Work with your tools until a minimal source fix is in place, then call "
            "submit_patch(). The working tree is graded even if you never call it, so "
            "never revert a fix you believe in. Never edit test files or pyproject.toml; "
            "they are reset before grading. Find the cause with git grep, confirm it by "
            "reading a narrow line range, make the smallest edit that fixes the reported "
            "behaviour, check it with python3 -m py_compile, and redirect any long "
            "output to /tmp with run_command so you can tail it. Submit before your "
            "budget runs out; a best guess beats no patch.\n"
            "{problem_description}\n"
        ),
    }
    stream = io.BytesIO()
    with zipfile.ZipFile(stream, "w", zipfile.ZIP_DEFLATED) as archive:
        for _path in sorted(emergency):
            _info = zipfile.ZipInfo(_path, date_time=(1980, 1, 1, 0, 0, 0))
            _info.compress_type = zipfile.ZIP_DEFLATED
            _info.create_system = 3
            _info.external_attr = 0o644 << 16
            archive.writestr(_info, emergency[_path].encode("utf-8"))
    submission_bytes = stream.getvalue()

SUBMISSION_PATH.write_bytes(submission_bytes)
print(f"wrote {SUBMISSION_PATH} ({len(submission_bytes):,d} bytes)")


In [ ]:
# A manifest makes the artifact auditable after the fact: what was chosen, why,
# what is inside, and the digest that pins the exact bytes being submitted.
try:
    with zipfile.ZipFile(SUBMISSION_PATH) as archive:
        members = {}
        for name in archive.namelist():
            raw = archive.read(name)
            members[name] = {
                "bytes": len(raw),
                "sha256": hashlib.sha256(raw).hexdigest(),
            }
        corrupt = archive.testzip()

    manifest = {
        "submission": SUBMISSION_PATH.name,
        "selected_package": SELECTED_NAME,
        "selection_reason": SELECTION_REASON,
        "submission_sha256": digest(submission_bytes),
        "submission_bytes": len(submission_bytes),
        "zip_crc_check": "clean" if corrupt is None else f"corrupt member {corrupt}",
        "members": members,
        "model": MODEL,
        "adapters": [n for n in SELECTED_FILES if n.startswith("adapters/")],
        "unpacked_bytes": sum(len(v.encode("utf-8")) for v in SELECTED_FILES.values()),
    }
    MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, sort_keys=True), encoding="utf-8")
    print(f"wrote {MANIFEST_PATH}")
except Exception:  # noqa: BLE001
    print("manifest could not be written (it is advisory only):")
    print(traceback.format_exc())


In [ ]:
# Final report. Reading the artifact back from disk is the last line of defence:
# these assertions check the file that actually exists, not the bytes in memory.
with zipfile.ZipFile(SUBMISSION_PATH) as archive:
    names = archive.namelist()
    assert archive.testzip() is None, "the written archive is corrupt"
    assert "agent.yaml" in names, "agent.yaml is not at the archive root"
    assert not any(n.endswith("/") for n in names), "the archive has directory entries"
    assert len(names) == len(set(names)), "the archive has duplicate names"
    for name in names:
        assert archive.read(name).decode("utf-8") == SELECTED_FILES[name], (
            f"{name} did not round trip byte-identically"
        )

print()
print("=" * 68)
print(f"  selected package : {SELECTED_NAME}")
print(f"  reason           : {SELECTION_REASON}")
print(f"  archive          : {SUBMISSION_PATH.name}  ({len(submission_bytes):,d} bytes)")
print(f"  sha256           : {digest(submission_bytes)}")
print(f"  members          : {len(names)}")
for name in names:
    print(f"      {name}")
print("=" * 68)
print("submission.zip is written and verified")

# An explicit sentinel, so a log scan can distinguish "finished and verified"
# from "the kernel died part way through" without having to parse the report.
print("DONE_OK")


### Done

`/kaggle/working/submission.zip` holds a declarative ADK agent package that the
organizers can compile. It declares a single model, ships no adapters, respects the
3 GiB and extension limits, and has a valid `agent.yaml` at its root.
